In [15]:
%pip install lightgbm imbalanced-learn scikit-learn joblib pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os

os.makedirs('../outputs/models', exist_ok=True)
print("Folders created.")

Folders created.


In [17]:
import os
import pandas as pd
import joblib

from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer


def build_pipeline():

    # Load fused dataset
    df = pd.read_csv('../data/processed/fused.csv')

    TARGET = 'target'

    DROP_COLS = [
        TARGET,
        'TWF',
        'HDF',
        'PWF',
        'OSF',
        'RNF'
    ]

    DROP_COLS = [
        c for c in DROP_COLS
        if c in df.columns
    ]

    # Features and target
    X = df.drop(columns=DROP_COLS)
    y = df[TARGET]

    # SMOTE + LightGBM pipeline
    pipe = ImbPipeline([
        (
            'smote',
            SMOTE(
                random_state=42,
                k_neighbors=3
            )
        ),
        (
            'clf',
            LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                num_leaves=31,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    # Cross-validation
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    macro_f1 = make_scorer(
        f1_score,
        average='macro'
    )

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=skf,
        scoring=macro_f1,
        n_jobs=-1
    )

    print(
        f"5-Fold Macro F1: "
        f"{scores.mean():.4f} ± {scores.std():.4f}"
    )

    # Train on full data
    pipe.fit(X, y)

    # Create folder if needed
    os.makedirs('../outputs/models', exist_ok=True)

    # Save model
    model_path = '../outputs/models/lgbm_pipeline.pkl'

    joblib.dump(
        pipe,
        model_path
    )

    print(f"Model saved at: {model_path}")

    return pipe, X, y


if __name__ == "__main__":
    pipe, X, y = build_pipeline()

5-Fold Macro F1: 0.9032 ± 0.0183
[LightGBM] [Info] Number of positive: 9661, number of negative: 9661
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007890 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14775
[LightGBM] [Info] Number of data points in the train set: 19322, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Model saved at: ../outputs/models/lgbm_pipeline.pkl


In [20]:
import os

print(os.path.exists('../outputs/models/lgbm_pipeline.pkl'))

True
